# Workflows in Agent Framework

**AI Agents** are dynamic - the LLM decides what steps to take at runtime based on context and available tools. Use when you need flexibility, reasoning, and open-ended problem solving.

**Workflows** are predefined sequences with explicit execution paths. The structure is fixed at design time, but can include conditional branching and parallel execution. Use when you need predictability, auditability, or coordination across multiple agents and external systems.

Workflows can contain agents as components - giving you LLM intelligence within a controlled structure.

### Core Concepts
1. **Executors**: Processing units that receive messages, perform tasks, and produce outputs. Can be agents or custom logic.
2. **Edges**: Connections between executors that control message flow. Support conditional routing.
3. **Workflows**: Directed graphs of executors and edges with a defined start point.

### Setup

First, let's validate our environment to ensure all required variables are configured.

In [ ]:
import os
from dotenv import load_dotenv
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

load_dotenv(override=True)

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)
print("Client ready.")

### Basic Executor Structure
Executors are the building blocks that process messages. They inherit from `Executor` and use the `@handler` decorator to define message handlers.

**Handler signature**: `(self, input: T_In, ctx: WorkflowContext[T_Out]) -> None`
- First param: the typed input message
- Second param: context for sending outputs downstream

#### Pattern 1: Class-based Executor
Here's an executor that validates an order and calculates the total:

In [ ]:
from agent_framework import Executor, WorkflowContext, handler

class ValidateOrder(Executor):
    def __init__(self):
        super().__init__(id="validate_order")
    
    @handler
    async def handle(self, order: dict, ctx: WorkflowContext[dict]) -> None:
        # Add validation status and calculate total
        order["is_valid"] = order.get("quantity", 0) > 0
        order["total"] = order.get("quantity", 0) * order.get("unit_price", 0)
        await ctx.send_message(order)

#### Pattern 2: Function-based Executor
For simple, stateless executors, use the `@executor` decorator on an async function.

In [ ]:
from agent_framework import executor, WorkflowContext

@executor(id="apply_discount")
async def apply_discount(order: dict, ctx: WorkflowContext[dict]) -> None:
    # Apply 10% discount for orders over $100
    if order.get("total", 0) > 100:
        order["discount"] = order["total"] * 0.10
        order["final_total"] = order["total"] - order["discount"]
    else:
        order["discount"] = 0
        order["final_total"] = order["total"]
    await ctx.send_message(order)

### WorkflowContext Methods
`WorkflowContext` provides two key methods for handlers:

| Method | Purpose | Context Type |
|--------|---------|-------------|
| `send_message(value)` | Forward to downstream executors | `WorkflowContext[T_Out]` |
| `yield_output(value)` | Emit as final workflow output | `WorkflowContext[Never, T_WorkflowOut]` |

If a handler does neither, use `WorkflowContext` with no type params.

**What is `Never`?** The `Never` type (from `typing_extensions`) indicates a **terminal executor** - the last step in a workflow branch. Terminal executors don't pass data downstream to other executors; instead, they produce the workflow's final output. Using `Never` as the first type parameter tells the type checker that `send_message()` should not be called.

Learn more about [`WorkflowContext` here](https://learn.microsoft.com/en-us/python/api/agent-framework-core/agent_framework.workflowcontext?view=agent-framework-python-latest)

In [ ]:
from agent_framework import Executor, WorkflowContext, handler
from typing_extensions import Never

class OrderProcessor(Executor):
    def __init__(self):
        super().__init__(id="order_processor")
    
    @handler
    async def forward_to_next(self, order: dict, ctx: WorkflowContext[dict]) -> None:
        """Send to downstream executor."""
        order["status"] = "processing"
        await ctx.send_message(order)

    @handler
    async def complete_order(self, order: dict, ctx: WorkflowContext[Never, str]) -> None:
        """Yield as workflow output (terminal node)."""
        await ctx.yield_output(f"Order {order['id']} completed. Total: ${order['final_total']:.2f}")

    @handler
    async def log_order(self, order: dict, ctx: WorkflowContext) -> None:
        """No output - just perform side effects."""
        print(f"Logging order {order['id']} to audit trail")

### Building Workflows with Edges

A **Workflow** connects executors and runs them. Use `WorkflowBuilder` to define the graph. The `start_executor` parameter tells the builder where to begin; your input goes to that executor first, which processes it and passes results downstream via edges.

**Edges** define how messages flow between executors. The framework supports:

- **Direct**: Simple one-to-one connections
- **Conditional**: Route based on message content
- **Fan-out**: One source to multiple targets
- **Fan-in**: Multiple sources to one target

There are lots of [examples on edges in AF](https://learn.microsoft.com/en-us/agent-framework/workflows/edges?pivots=programming-language-python), make sure to check them out!

**Conditional edges** allow your workflow to make routing decisions based on the content or properties of messages flowing through the workflow. This enables dynamic branching where different execution paths are taken based on runtime conditions.

Let's build an order processing workflow with conditional routing:

In [ ]:
from agent_framework import WorkflowBuilder, WorkflowContext, executor
from typing_extensions import Never

@executor(id="validate_order")
async def validate_order(order: dict, ctx: WorkflowContext[dict]) -> None:
    """Validate and enrich the order."""
    order["is_valid"] = order.get("quantity", 0) > 0
    order["total"] = order.get("quantity", 0) * order.get("unit_price", 0)
    await ctx.send_message(order)

@executor(id="standard_processing")
async def standard_processing(order: dict, ctx: WorkflowContext[Never, str]) -> None:
    """Process standard orders."""
    await ctx.yield_output(f"Standard order processed: ${order['total']:.2f}")

@executor(id="priority_processing")
async def priority_processing(order: dict, ctx: WorkflowContext[Never, str]) -> None:
    """Process high-value orders with priority handling."""
    discount = order["total"] * 0.05  # 5% loyalty discount
    await ctx.yield_output(f"Priority order processed: ${order['total']:.2f} (loyalty discount: ${discount:.2f})")

# Build workflow with conditional routing based on order value
workflow = (
    WorkflowBuilder(start_executor=validate_order)
    .add_edge(validate_order, standard_processing)  # Direct: always executes
    .add_edge(validate_order, priority_processing, condition=lambda order: order.get("total", 0) > 500)  # Conditional
    .build()
)

# Test with small order (only standard_processing runs)
small_order = {"id": "ORD-001", "quantity": 2, "unit_price": 25.00}
print(f"Small order: {small_order}")
result = await workflow.run(small_order)
print(f"Results: {result.get_outputs()}")

print()

# Test with large order (both processors run - fan-out)
large_order = {"id": "ORD-002", "quantity": 50, "unit_price": 25.00}
print(f"Large order: {large_order}")
result = await workflow.run(large_order)
print(f"Results: {result.get_outputs()}")

### Using Agents in Workflows

You can pass `Agent` instances directly to a workflow. The framework treats them as executors (internally wrapping them to integrate with the workflow execution model).

When you add an agent to a workflow:
- It behaves like an executor, processing inputs according to its instructions and context.
- You can chain multiple agents together to create collaborative flows (e.g., Writer → Reviewer).
- The workflow orchestrates their interaction by routing messages between agents and exposing outputs and events to the caller.

Workflow execution is event-driven: during a run, events (such as `executor_invoked`, `executor_completed`, and `output`) provide observability into each step of the pipeline.

In the following example, two agents are created and connected in a simple pipeline:

In [ ]:
from agent_framework import Agent, WorkflowBuilder, WorkflowEvent

writer_agent = Agent(
    client=client,
    name="writer",
    instructions="You are an excellent content writer. You create new content and edit contents based on the feedback.",
)

reviewer_agent = Agent(
    client=client,
    name="reviewer",
    instructions=(
        "You are an excellent content reviewer. "
        "Provide actionable feedback to the writer about the provided content. "
        "Provide the feedback in the most concise manner possible."
    ),
)

# Build workflow - agents can be passed directly to add_edge and start_executor
workflow = (
    WorkflowBuilder(start_executor=writer_agent)
    .add_edge(writer_agent, reviewer_agent)
    .build()
)

# Run the workflow
result = await workflow.run("Create a slogan for a new electric SUV that is affordable and fun to drive.")

# Print events to inspect execution details
for event in result:
    if event.type == "executor_completed":
        print(f"\nExecutor: {event.executor_id}")
        print(f"Output:\n{event.data}")
        print("-" * 40)


outputs = result.get_outputs()

print("\n--- Final Outputs ---")
for output in outputs:
    print(output)


print(f"\nWorkflow State: {result.get_final_state()}")

#### Understanding Workflow Events

When you run a workflow with `workflow.run()`, it returns a `WorkflowRunResult` - a list of `WorkflowEvent` objects that you can iterate and inspect.

Workflow events are split conceptually into:
  - **Data-plane events** → executor activity, outputs
  - **Control-plane events** → workflow state (`status` events)


All events use a single `WorkflowEvent` class with an `event_type` field that identifies the kind of event:


| `type` | Description |
|---|---|
| `"started"` | Workflow execution started |
| `"status"` | Workflow state changed (use `event.state`) |
| `"executor_invoked"` | An executor started processing |
| `"executor_completed"` | An executor finished processing |
| `"executor_failed"` | An executor encountered an error |
| `"output"` | A workflow-level output was produced |
| `"failed"` | The entire workflow failed |
| `"error"` | Non-fatal error from user code |
| `"warning"` | Non-fatal warning from user code |


**`WorkflowEvent` Properties:**

| Property | Type | Description |
|---|---|---|
| `type` | `str` | The type of event (see table above) |
| `data` | `Any` | The event payload (agent response, error details, etc.) |

**Result Methods:**

| Method | Returns | Description |
|---|---|---|
| `get_outputs()` | `list` | Extract all workflow outputs (preferred way to get final results) |
| `get_final_state()` | `WorkflowRunState` | The workflow's final state (e.g., `IDLE`, `FAILED`) |

This pattern is essential for debugging multi-agent workflows - you can see exactly what each agent produced and trace the flow of information through your pipeline. 

Learn more about [`WorkflowRunResult`](https://learn.microsoft.com/en-us/python/api/agent-framework-core/agent_framework.workflowrunresult?view=agent-framework-python-latest), [Workflow Events](https://learn.microsoft.com/en-us/agent-framework/workflows/events?pivots=programming-language-python) and [`WorkflowRunState`](https://learn.microsoft.com/en-us/agent-framework/workflows/events?pivots=programming-language-python)

### Exercise 1 - Create a Multi-Agent Workflow

Open **`03-tasks.ipynb`** and complete the exercise 1. You will your own workflow with agents that collaborate on a task.

### Custom Executors

So far we've passed agents directly to `WorkflowBuilder`, which auto-wraps them in `AgentExecutor`. But sometimes you need more control - custom input/output handling, preprocessing, or managing conversation history between agents.

**Custom executors** let you wrap agents in your own classes. You subclass `Executor`, create the agent in `__init__`, and define a `@handler` method to control how messages flow.

Here's a Writer executor that:
- Receives a string prompt as input
- Runs its internal agent
- Passes the agent's response text downstream

In [ ]:
from agent_framework import Agent, Executor, WorkflowContext, handler, Message
from typing_extensions import Never

class Writer(Executor):
    """Custom executor that owns a writer agent."""

    def __init__(self, chat_client, id: str = "writer"):
        self.agent = Agent(
            client=chat_client,
            name="writer",
            instructions="You are an excellent content writer. You create new content and edit contents based on the feedback.",
        )
        super().__init__(id=id)

    @handler
    async def handle(self, prompt: str, ctx: WorkflowContext[str]) -> None:
        """Generate content and forward the response text downstream."""
        response = await self.agent.run(prompt)
        await ctx.send_message(response.text)

And a Reviewer executor that:
- Receives the writer's output as a string
- Reviews the content
- Yields the final output (terminal node)

In [ ]:
class Reviewer(Executor):
    """Custom executor that owns a reviewer agent."""

    def __init__(self, chat_client, id: str = "reviewer"):
        self.agent = Agent(
            client=chat_client,
            name="reviewer",
            instructions="You are an excellent content reviewer. Review the content and provide constructive feedback.",
        )
        super().__init__(id=id)

    @handler
    async def handle(self, content: str, ctx: WorkflowContext[Never, str]) -> None:
        """Review content and yield final output."""
        response = await self.agent.run(content)
        await ctx.yield_output(response.text)

Now let's build and run the workflow with these custom executors.

Since these are stateful classes (each holds its own agent), we create instances and pass them directly:

In [ ]:
from agent_framework import WorkflowBuilder

writer = Writer(client)
reviewer = Reviewer(client)

# Build workflow

workflow = (
    WorkflowBuilder(start_executor=writer)
    .add_edge(writer, reviewer)
    .build()
)


# Run the workflow
result = await workflow.run(
    "Create a slogan for a new electric SUV that is affordable and fun to drive."
)


outputs = result.get_outputs()

print("Output:", outputs[0] if outputs else "No output")
print("State:", result.get_final_state())


### Exercise 2: Build a Custom Executor Workflow with Streaming

Now go to **`03-tasks.ipynb`** and complete the exercise 2. You will create a workflow with custom executors that's able to process support tickets.

### Streaming Workflow Events

For real-time observability, use `workflow.run(..., stream=True)` instead of the default non-streaming mode. This returns a `ResponseStream` that yields `WorkflowEvent` objects as they happen:

| `type` | Description |
|---|---|
| `"status"` | State changes (IN_PROGRESS, IDLE, etc.) |
| `"output"` | Final outputs from terminal executors |
| `"executor_failed"` | An executor encountered an error |
| `"failed"` | The entire workflow failed |

Learn more about [`WorkflowEvent`](https://learn.microsoft.com/en-us/python/api/agent-framework-core/agent_framework.workflowevent?view=agent-framework-python-latest) and  [`Events`](https://learn.microsoft.com/en-us/agent-framework/workflows/events?pivots=programming-language-python). Below we provide an example of how you can consume different events. Feel free to experiment and stream other information from the documentation!

In [ ]:
from agent_framework import WorkflowBuilder, WorkflowEvent

# Rebuild workflow with fresh instances
writer = Writer(client)
reviewer = Reviewer(client)

workflow = (
    WorkflowBuilder(start_executor=writer)
    .add_edge(writer, reviewer)
    .build()
)

# Stream events
async for event in workflow.run(
    "Create a slogan for a new electric SUV that is affordable and fun to drive.",
    stream=True,
):
    if event.type == "started":
        print(f"Workflow started: {event.executor_id}")
    elif event.type == "output":
        print(f"Output: {event.data}")
    elif event.type == "executor_failed":
        print(f"Executor failed: {event.source.executor_id} - {event.data}")
    elif event.type == "executor_invoked":
        print(f"Executor invoked: {event.data}")

---
## Recap and what's next

In this notebook, you learned the fundamentals of building workflows with the Microsoft Agent Framework:

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Executor** | A processing unit that receives typed messages, performs operations, and produces outputs |
| **WorkflowContext** | Provides methods (`send_message`, `yield_output`) for handlers to interact with the workflow |
| **Edges** | Define connections between executors with optional conditions for routing |
| **WorkflowBuilder** | Fluent API for constructing workflows by connecting executors |
| **WorkflowEvent** | A unified event type with `event_type` field for monitoring execution |

### What You Built

1. **Basic Executors** - Class-based and function-based executors with `@handler` decorators
2. **Sequential Workflows** - Linear pipelines connecting multiple executors
3. **Conditional Workflows** - Fan-out patterns with condition-based routing
4. **Agent Workflows** - AI agents passed directly to `WorkflowBuilder`
5. **Custom Agent Executors** - Wrapped agents with full control over message flow


### Next Steps

In the next notebook, you'll learn about **pre-built orchestration patterns** including Sequential, Concurrent, and Handoff orchestrations that simplify multi-agent coordination.